<a href="https://colab.research.google.com/github/Dadi-Lakshmi-Harsha/GPU_Programminig/blob/main/GPU_assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%writefile graph.txt
5 6
0 1 10
0 2 3
1 3 2
2 1 1
2 3 8
3 4 5

Overwriting graph.txt


In [9]:
%%writefile assigment2.cu
#include <iostream>
#include <fstream>
#include <vector>
using namespace std;
struct Edge
{
    int source;
    int destination;
    int weight;
};


int main()
{
    ifstream file("graph.txt");

    if (!file)
    {
        cout <<"Error opening graph.txt"<<endl;
        return 1;
    }

    int V, E;

    file >> V >> E;

    vector<Edge> edges(E);

    for (int i = 0; i < E; i++)
    {
        file >> edges[i].source
             >> edges[i].destination
             >> edges[i].weight;
    }

    file.close();
    vector<int> rowPtr(V + 1, 0);
    vector<int> col(E);
    vector<int> weight(E);


    // Count outgoing edges
    for (int i = 0; i < E; i++)
    {
        rowPtr[edges[i].source + 1]++;
    }


    // Convert counts into positions
    for (int i = 1; i <= V; i++)
    {
        rowPtr[i] += rowPtr[i - 1];
    }


    // Temporary positions
    vector<int> position = rowPtr;


    // Fill CSR arrays
    for (int i = 0; i < E; i++)
    {
        int source = edges[i].source;

        col[position[source]] =
            edges[i].destination;

        weight[position[source]] =
            edges[i].weight;

        position[source]++;
    }

    ofstream output("csr.txt");
    if (!output)
    {
        cout<<"Error creating csr.txt" << endl;
        return 1;
    }
    // Number of vertices and edges
    output << V << " " << E << endl;
    for (int i = 0; i <= V; i++)
    {
        output << rowPtr[i] << " ";
    }
    output << endl;
    for (int i = 0; i < E; i++)
    {
        output << col[i] << " ";
    }
    output << endl;
    for (int i = 0; i < E; i++)
    {
        output << weight[i] << " ";
    }
    output << endl;
    output.close();
    cout << "CSR Format" << endl;
    cout << "row_ptr: ";
    for (int x : rowPtr)
        cout << x << " ";
    cout << endl;
    cout << "col:     ";
    for (int x : col)
        cout << x << " ";
    cout << endl;
    cout << "weight:  ";
    for (int x : weight)
        cout << x << " ";
    cout << endl;
    cout << "\nCSR saved to csr.txt" << endl;
    return 0;
}

Overwriting assigment2.cu


In [10]:
!nvcc -o assigment2 assigment2.cu
!./assigment2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CSR Format
row_ptr: 0 2 3 5 6 6 
col:     1 2 3 1 3 4 
weight:  10 3 2 1 8 5 

CSR saved to csr.txt


In [12]:
%%writefile assigment2_1.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <climits>
#include <cuda_runtime.h>

using namespace std;

#define INF INT_MAX
__global__ void ssspKernel(
    int V,
    const int *rowPtr,
    const int *col,
    const int *weight,
    int *dist,
    int *changed)
{
    int u=blockIdx.x * blockDim.x + threadIdx.x;
    if (u >= V)
        return;
    // Skip unreachable vertices
    if (dist[u] == INF)
        return;
    // Process all neighbours of vertex u
    for (int i = rowPtr[u]; i < rowPtr[u + 1]; i++)
    {
        int v=col[i];
        int w=weight[i];
        int newDist=dist[u] + w;
        if (newDist < dist[v])
        {
            atomicMin(&dist[v], newDist);
        }
    }
}

int main()
{
    ifstream file("csr.txt");

    if (!file)
    {
        cout << "Error opening csr.txt" << endl;
        return 1;
    }
    int V, E;
    file >>V >> E;
    vector<int> h_rowPtr(V + 1);
    vector<int> h_col(E);
    vector<int> h_weight(E);

    // Read row_ptr
    for (int i = 0;i<=V;i++)
        file>>h_rowPtr[i];
    // Read col
    for (int i=0;i<E;i++)
        file>>h_col[i];
    // Read weight
    for (int i = 0; i<E;i++)
        file>>h_weight[i];
    file.close();
    int source;
    cout << "Enter source vertex: ";
    cin >> source;

    if (source < 0 || source >= V)
    {
        cout << "Invalid source vertex" << endl;
        return 1;
    }

    int *d_rowPtr;
    int *d_col;
    int *d_weight;
    int *d_dist;
    int *d_changed;

    cudaMalloc(&d_rowPtr, (V + 1) * sizeof(int));
    cudaMalloc(&d_col, E * sizeof(int));
    cudaMalloc(&d_weight, E * sizeof(int));
    cudaMalloc(&d_dist, V * sizeof(int));
    cudaMalloc(&d_changed, sizeof(int));
    cudaMemcpy(
        d_rowPtr,
        h_rowPtr.data(),
        (V + 1) * sizeof(int),
        cudaMemcpyHostToDevice);
    cudaMemcpy(
        d_col,
        h_col.data(),
        E * sizeof(int),
        cudaMemcpyHostToDevice);
    cudaMemcpy(
        d_weight,
        h_weight.data(),
        E * sizeof(int),
        cudaMemcpyHostToDevice);
    vector<int> h_dist(V, INF);
    h_dist[source] = 0;
    cudaMemcpy(
        d_dist,
        h_dist.data(),
        V * sizeof(int),
        cudaMemcpyHostToDevice);
    int threadsPerBlock = 256;
    int blocksPerGrid =
        (V + threadsPerBlock - 1) / threadsPerBlock;
    int changed = 1;
    while (changed)
    {
        // Reset changed flag
        changed = 0;

        cudaMemcpy(
            d_changed,
            &changed,
            sizeof(int),
            cudaMemcpyHostToDevice);
        // Launch kernel
        ssspKernel<<<blocksPerGrid, threadsPerBlock>>>(
            V,
            d_rowPtr,
            d_col,
            d_weight,
            d_dist,
            d_changed);
        // Wait for GPU
        cudaDeviceSynchronize();
        // Check whether any distance changed
        cudaMemcpy(
            &changed,
            d_changed,
            sizeof(int),
            cudaMemcpyDeviceToHost);
    }
    cudaMemcpy(
        h_dist.data(),
        d_dist,
        V * sizeof(int),
        cudaMemcpyDeviceToHost);
    cout << endl;
    cout << "GPU SSSP Result" << endl;

    for (int i = 0; i < V; i++)
    {
        cout << "Vertex " << i << " : ";
        if (h_dist[i] == INF)
            cout << "INF";
        else
            cout<<h_dist[i];
        cout<<endl;
    }

    cudaFree(d_rowPtr);
    cudaFree(d_col);
    cudaFree(d_weight);
    cudaFree(d_dist);
    cudaFree(d_changed);

    return 0;
}

Writing assigment2_1.cu


In [14]:
!nvcc -o assigment2 assigment2_1.cu
!./assigment2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter source vertex: 0

GPU SSSP Result
Vertex 0 : 0
Vertex 1 : 4
Vertex 2 : 3
Vertex 3 : 6
Vertex 4 : 11


In [15]:
%%writefile assigment2_cpu.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <climits>
using namespace std;
#define INF INT_MAX
int main()
{
    ifstream file("csr.txt");
    if (!file)
    {
        cout <<"Error opening csr.txt" << endl;
        return 1;
    }

    int V, E; // V: number of vertices, E: number of edges
    file >> V >> E;
    vector<int> rowPtr(V + 1);
    vector<int> col(E);
    vector<int> weight(E);
    // Read row_ptr
    for (int i=0; i<=V;i++)
        file >> rowPtr[i];
    // Read col
    for (int i = 0; i < E; i++)
        file >> col[i];

    // Read weight
    for (int i = 0;i<E;i++)
        file>>weight[i];
    file.close();
    int source;
    cout<<"Enter source vertex: ";
    cin>>source;
    if (source<0||source>=V)
    {
        cout << "Invalid source vertex" << endl;
        return 1;
    }
    vector<int> dist(V, INF); // Initialize distances to infinity
    dist[source] = 0;         // Distance to source is 0
    for (int k = 0; k < V - 1; k++)
    {
        bool changed=false;
        for (int u=0;u<V;u++)
        {
            if (dist[u] == INF) continue;
            for (int i = rowPtr[u]; i < rowPtr[u + 1]; i++)
            {
                int v = col[i];
                int w = weight[i];
                if (dist[u] != INF && dist[u] + w < dist[v])
                {
                    dist[v] = dist[u] + w;
                    changed = true;
                }
            }
        }
        if (!changed) break;
    }
    cout << endl;
    cout << "CPU SSSP Result" << endl;
    for (int i=0;i<V;i++)
    {
        cout<<"Vertex"<<i<< " : ";
        if (dist[i]==INF)
            cout<<"INF";
        else
            cout<<dist[i];
        cout<<endl;
    }
    return 0;
}

Writing assigment2_cpu.cu


In [16]:
!nvcc -o assigment2_cpu assigment2_cpu.cu
!./assigment2_cpu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter source vertex: 0

CPU SSSP Result
Vertex 0 : 0
Vertex 1 : 4
Vertex 2 : 3
Vertex 3 : 6
Vertex 4 : 11
